# 장갑 보정 키보드 — 본 실험 분석 노트북

이 노트북은 분석 스크립트 두 개를 순서대로 실행하고, 결과 표와 그래프를 보여 준다.

1. `extract.mjs` (Node): 원본 로그(JSONL)에서 참가자 × 조건별 지표를 다시 계산한다. → `analysis_out/per_condition.csv`
2. `stats.py` (Python): 2×2 반복측정 분산분석, ART, 사전 지정 비교(H1·H4)를 하고 그래프를 만든다. → `analysis_out/report.md`, `fig1–3.png`

각 통계 기법을 왜 쓰는지는 `stats.py`의 주석과 `analysis/README.md`에 정리돼 있다.

**준비물**
- 이 저장소
- 데이터 폴더 `실험 결과 데이터/`. 저장소 맨 위에 두며, 공개 저장소에는 없으므로 팀 내부에서 따로 받는다.
- Node 18 이상
- numpy, scipy, matplotlib, pandas

> ⚠️ **커밋 전에 반드시 출력 지우기.** 실행 결과(표·그래프)에는 참가자 데이터가 들어 있다. 이 노트북을 공개 저장소에 다시 올릴 때는 메뉴에서 *Edit → Clear Outputs of All Cells* 를 한 뒤 저장한다.

In [ ]:
# 1) 경로 설정: 노트북을 analysis/ 폴더에서 열든 저장소 맨 위에서 열든 동작하게 한다.
import os, sys, subprocess
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
os.chdir(ROOT)
DATA = ROOT / "실험 결과 데이터"   # 다른 곳에 있으면 이 경로만 바꾼다
print("저장소:", ROOT)
print("데이터 폴더 있음:", DATA.is_dir())

In [ ]:
# 2) 원본 로그 → 정리 표 (Node 스크립트 실행)
#    경고 0건이어야 정상(문장 수 8개, 다시 만든 입력 = 기록된 입력).
r = subprocess.run(["node", "analysis/extract.mjs", str(DATA)], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(open("analysis_out/extract_warnings.txt", encoding="utf-8").read().strip() or "(경고 없음)")

In [ ]:
# 3) 통계 분석 + 그래프 (지금 쓰는 파이썬으로 stats.py 실행) → 결과 표를 아래에 보여 준다.
from IPython.display import Markdown, Image, display

r = subprocess.run([sys.executable, "-I", "analysis/stats.py"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr)
display(Markdown(open("analysis_out/report.md", encoding="utf-8").read()))

In [ ]:
# 4) 그래프 세 개
for f in ["fig1_interaction.png", "fig2_participants.png", "fig3_differences.png"]:
    display(Markdown(f"**{f}**"))
    display(Image(filename=f"analysis_out/{f}"))

## 직접 더 살펴보기 (선택)

아래는 정리 표를 pandas로 불러와 조건별 평균을 보는 예시다. 다른 지표를 보려면 열 이름만 바꾸면 된다.

**열 이름 목록:**
- 오류·속도: `neighbor_error_rate`, `key_error_rate`, `wpm`, `backspaces`, `msd_rate`, `omission_rate`
- 섀도 판정: `shadow_base_neighbor_error_rate`, `shadow_cal_neighbor_error_rate`
- 설문: `tlx_raw`, `discomfort`, `sus`, `pref_glove`, `pref_bare`

In [ ]:
import pandas as pd

df = pd.read_csv("analysis_out/per_condition.csv")
df.pivot_table(index="cond", values=["neighbor_error_rate", "wpm", "tlx_raw", "sus"], aggfunc=["mean", "std"]).round(3)